In [2]:
# ══════════════════════════════════════════════════════════════════════
# CELL I1 — INbreast DISCOVERY + INVENTORY  (external validation set)
#   Auto-finds AllDICOMs / AllXML / INbreast.csv wherever they sit,
#   parses the OsiriX plist annotations, builds the mass manifest.
#   CPU only, ~30 s.
# ══════════════════════════════════════════════════════════════════════
import os, re, glob, plistlib, collections
import numpy as np, pandas as pd
pd.set_option("display.width", 200)

SEARCH_ROOTS = ["/root/autodl-tmp/INBreast_data", "/root/autodl-tmp/INbreast_data",
                "/root/autodl-tmp", "/root", "."]
OUT = "/root/autodl-tmp/INB"; os.makedirs(OUT, exist_ok=True)

# ---------- 1. discover the pieces -------------------------------------
def walk(root, maxdepth=6):
    root = os.path.abspath(root)
    base = root.rstrip("/").count("/")
    for dp, dn, fn in os.walk(root):
        if dp.count("/") - base > maxdepth:
            dn[:] = []; continue
        dn[:] = [d for d in dn if not d.startswith(".")]
        yield dp, fn

dcm_dirs, xml_dirs, csv_hits, xls_hits = collections.Counter(), collections.Counter(), [], []
seen = set()
for r in SEARCH_ROOTS:
    if not os.path.isdir(r) or os.path.abspath(r) in seen: continue
    seen.add(os.path.abspath(r))
    for dp, fn in walk(r):
        nd = sum(1 for f in fn if f.lower().endswith(".dcm"))
        nx = sum(1 for f in fn if f.lower().endswith(".xml"))
        if nd: dcm_dirs[dp] += nd
        if nx: xml_dirs[dp] += nx
        for f in fn:
            lf = f.lower()
            if lf == "inbreast.csv": csv_hits.append(os.path.join(dp, f))
            if lf in ("inbreast.xls", "inbreast.xlsx"): xls_hits.append(os.path.join(dp, f))
    if dcm_dirs and (csv_hits or xls_hits): break

print("="*84); print("DISCOVERY"); print("="*84)
print("  DICOM dirs found:")
for d, n in dcm_dirs.most_common(5): print("     %5d files  %s" % (n, d))
print("  XML dirs found:")
for d, n in xml_dirs.most_common(5): print("     %5d files  %s" % (n, d))
print("  INbreast.csv :", csv_hits[:2] or "NOT FOUND")
print("  INbreast.xls :", xls_hits[:2] or "NOT FOUND")

assert dcm_dirs, ("No .dcm files found under %s — set SEARCH_ROOTS to the folder "
                  "containing your INbreast data." % SEARCH_ROOTS)
DCM_DIR = dcm_dirs.most_common(1)[0][0]
XML_DIR = xml_dirs.most_common(1)[0][0] if xml_dirs else None
print("\n  using DICOMs from : %s" % DCM_DIR)
print("  using XMLs   from : %s" % (XML_DIR or "NONE — segmentation cannot be validated"))

try:
    import pydicom; print("  pydicom           : %s" % pydicom.__version__)
except ImportError:
    print("  pydicom           : MISSING  ->  pip install pydicom")

# ---------- 2. case table (BI-RADS labels) -----------------------------
meta = None
if csv_hits:
    for sep in [";", ","]:
        try:
            t = pd.read_csv(csv_hits[0], sep=sep, encoding="latin-1")
            if t.shape[1] >= 6: meta = t; break
        except Exception: pass
if meta is None and xls_hits:
    try: meta = pd.read_excel(xls_hits[0])
    except Exception as e: print("  could not read xls:", e)
assert meta is not None, "INbreast.csv / .xls not found — needed for BI-RADS labels."

meta.columns = [str(c).strip() for c in meta.columns]
ren = {}
for c in meta.columns:
    lc = c.lower()
    if "file" in lc and "name" in lc: ren[c] = "img_id"
    elif "bi-rads" in lc or "birads" in lc: ren[c] = "birads_raw"
    elif lc == "acr": ren[c] = "acr"
    elif "lateral" in lc: ren[c] = "side"
    elif lc == "view": ren[c] = "view"
meta = meta.rename(columns=ren)
need = {"img_id", "birads_raw"}
assert need <= set(meta.columns), "missing columns %s; found %s" % (need - set(meta.columns), list(meta.columns))
meta["img_id"] = meta["img_id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
meta = meta[meta["img_id"].str.len() > 0].copy()
meta["birads"] = meta["birads_raw"].apply(
    lambda s: int(re.match(r"\s*(\d)", str(s)).group(1)) if re.match(r"\s*(\d)", str(s)) else -1)
meta["label"] = (meta["birads"] >= 4).astype(int)          # 1-3 benign, 4-6 malignant
for c in ["acr", "side", "view"]:
    if c not in meta.columns: meta[c] = ""
print("\ncase table rows: %d   BI-RADS: %s"
      % (len(meta), dict(sorted(collections.Counter(meta['birads_raw'].astype(str)).items()))))

# ---------- 3. map image id -> DICOM path + patient --------------------
dcm = {}
for f in glob.glob(os.path.join(DCM_DIR, "**", "*.dcm"), recursive=True):
    b = os.path.basename(f); parts = b.split("_")
    iid = parts[0]
    pid = parts[1] if len(parts) >= 2 else iid
    dcm[iid] = (f, pid)
meta["dcm"] = meta["img_id"].map(lambda i: dcm.get(i, (None, None))[0])
meta["pid"] = meta["img_id"].map(lambda i: dcm.get(i, (None, None))[1])
print("DICOMs on disk: %d   matched to case table: %d" % (len(dcm), meta["dcm"].notna().sum()))
if meta["dcm"].notna().sum() == 0 and dcm:
    print("  !! no matches — sample dcm names:", list(dcm.keys())[:3],
          "| sample img_id:", meta["img_id"].head(3).tolist())

# ---------- 4. mass contours from the plist XMLs -----------------------
MASS_NAMES = {"mass", "masses"}
def parse_rois(path):
    try:
        with open(path, "rb") as fh: pl = plistlib.load(fh, fmt=plistlib.FMT_XML)
    except Exception:
        return []
    out = []
    for im in pl.get("Images", []):
        for roi in im.get("ROIs", []):
            pts = []
            for s in roi.get("Point_px", []):
                m = re.match(r"\(\s*([-\d.]+)\s*,\s*([-\d.]+)", str(s))
                if m: pts.append((float(m.group(1)), float(m.group(2))))
            out.append({"name": str(roi.get("Name", "")).strip(), "pts": pts})
    return out

xml_map, names = {}, collections.Counter()
if XML_DIR:
    for f in glob.glob(os.path.join(XML_DIR, "**", "*.xml"), recursive=True):
        iid = os.path.splitext(os.path.basename(f))[0]
        rois = parse_rois(f); xml_map[iid] = rois
        for r in rois: names[r["name"].lower()] += 1
print("XML files parsed: %d" % len(xml_map))
print("ROI names       :", dict(names.most_common(8)))

def mass_rois(i): return [r for r in xml_map.get(i, [])
                          if r["name"].lower() in MASS_NAMES and len(r["pts"]) >= 8]
meta["n_mass"] = meta["img_id"].apply(lambda i: len(mass_rois(i)))

# ---------- 5. the usable cohort ---------------------------------------
M = meta[(meta["n_mass"] > 0) & meta["dcm"].notna()].copy().reset_index(drop=True)
print("\n" + "="*84); print("USABLE EXTERNAL MASS COHORT"); print("="*84)
print("  images with a mass contour + DICOM : %d" % len(M))
print("  distinct patients                  : %d" % M["pid"].nunique())
print("  total mass contours                : %d" % int(M["n_mass"].sum()))
if len(M):
    print("  malignant (BI-RADS 4-6)            : %d (%.1f%%)" % (M["label"].sum(), 100*M["label"].mean()))
    t = (M.groupby("birads_raw").agg(images=("img_id","size"), patients=("pid","nunique"),
                                     lab=("label","first")).reset_index().sort_values("birads_raw"))
    t["class"] = np.where(t["lab"] == 1, "malignant", "benign")
    print("\n" + t[["birads_raw","images","patients","class"]].to_string(index=False))
    print("\n  views:", dict(M["view"].astype(str).value_counts()),
          " sides:", dict(M["side"].astype(str).value_counts()))
    w = []
    for _, r in M.iterrows():
        for roi in mass_rois(r["img_id"]):
            p = np.array(roi["pts"])
            w.append(max(p[:,0].max()-p[:,0].min(), p[:,1].max()-p[:,1].min()))
    w = np.array(w)
    print("\n  mass width (px): median %.0f  p10 %.0f  p90 %.0f  max %.0f"
          % (np.median(w), np.percentile(w,10), np.percentile(w,90), w.max()))

    M[["img_id","pid","side","view","birads_raw","birads","acr","label","n_mass","dcm"]] \
        .to_csv(os.path.join(OUT, "inbreast_manifest.csv"), index=False)
    np.save(os.path.join(OUT, "inbreast_mass_points.npy"),
            {r["img_id"]: [x["pts"] for x in mass_rois(r["img_id"])] for _, r in M.iterrows()},
            allow_pickle=True)
    print("\nsaved %s/inbreast_manifest.csv + inbreast_mass_points.npy" % OUT)
else:
    print("\n  !! nothing usable — paste the DISCOVERY block above and I'll fix the paths.")

print("\n" + "="*84)
print("LABEL DEFINITION — must be stated in the thesis")
print("="*84)
print("  INbreast gives BI-RADS, not biopsy pathology. We use BI-RADS 1-3 = benign,")
print("  4-6 = malignant. CBIS-DDSM labels are biopsy-confirmed, so this validates")
print("  agreement with radiologist assessment, not with histology. And because")
print("  BI-RADS is the label here, the risk-stratified threshold layer cannot be")
print("  used on INbreast — a single global threshold is applied instead.")

DISCOVERY
  DICOM dirs found:
       410 files  /root/autodl-tmp/INBreast_data/INbreast Release 1.0/AllDICOMs
  XML dirs found:
       343 files  /root/autodl-tmp/INBreast_data/INbreast Release 1.0/AllXML
       201 files  /root/autodl-tmp/INBreast_data/INbreast Release 1.0/PectoralMuscle/Pectoral Muscle XML
  INbreast.csv : ['/root/autodl-tmp/INBreast_data/INbreast Release 1.0/INbreast.csv']
  INbreast.xls : ['/root/autodl-tmp/INBreast_data/INbreast Release 1.0/INbreast.xls']

  using DICOMs from : /root/autodl-tmp/INBreast_data/INbreast Release 1.0/AllDICOMs
  using XMLs   from : /root/autodl-tmp/INBreast_data/INbreast Release 1.0/AllXML
  pydicom           : 3.0.2

case table rows: 410   BI-RADS: {'1': 67, '2': 220, '3': 23, '4a': 13, '4b': 8, '4c': 22, '5': 49, '6': 8}
DICOMs on disk: 410   matched to case table: 410
XML files parsed: 343
ROI names       : {'calcification': 7142, 'mass': 116, 'cluster': 27, 'spiculated region': 15, 'asymmetry': 5, 'distortion': 3, 'unnamed': 2, '':

In [3]:
# ══════════════════════════════════════════════════════════════════════
# CELL I2 — INbreast PREPROCESSING
#   DICOM -> 8-bit  |  polygons -> masks  |  tight + wide crops
#   Geometry copied exactly from the CBIS pipeline (Cell 29c).
#   CPU only, ~3 min.
# ══════════════════════════════════════════════════════════════════════
import os, re, glob, time
import numpy as np, pandas as pd, cv2, pydicom
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
cv2.setNumThreads(0)

OUT   = "/root/autodl-tmp/INB"
CROPT = os.path.join(OUT, "crops_tight"); os.makedirs(CROPT, exist_ok=True)
CROPW = os.path.join(OUT, "crops_wide");  os.makedirs(CROPW, exist_ok=True)
FIG   = os.path.join(OUT, "figures");     os.makedirs(FIG, exist_ok=True)

S           = 512      # same as CBIS crops
TARGET_COV  = 0.23     # CBIS median mask coverage of the tight crop
WIDE_MULT   = 1.75     # same wide factor as CBIS
MARGIN      = 0.10
rng = np.random.default_rng(0)

man = pd.read_csv(os.path.join(OUT, "inbreast_manifest.csv"))
man["img_id"] = man["img_id"].astype(str)
pts_map = np.load(os.path.join(OUT, "inbreast_mass_points.npy"), allow_pickle=True).item()
print("manifest rows: %d   images with points: %d" % (len(man), len(pts_map)))

# ---------- geometry helpers (identical to CBIS Cell 29c) --------------
def place(lo_obj, hi_obj, centre, s, limit, m):
    want = centre - s/2.0
    lo, hi = hi_obj + m - s, lo_obj - m
    if lo > hi: return int(round(want)), True
    lo2, hi2 = max(lo, 0.0), min(hi, limit - s)
    if lo2 > hi2: return int(round(min(max(want, lo), hi))), True
    return int(round(min(max(want, lo2), hi2))), False

def fill(st, shape):
    mu, sd = float(np.median(st)), float(st.std())
    return np.clip(rng.normal(mu, max(0.5*sd, 1.0), shape), 0, 255).astype(np.uint8)

def cut(im, y0, x0, s, interp, zero_pad=False):
    y1, x1 = y0 + s, x0 + s
    py0, px0 = max(0, -y0), max(0, -x0)
    py1, px1 = max(0, y1 - im.shape[0]), max(0, x1 - im.shape[1])
    sub = im[max(y0,0):min(y1,im.shape[0]), max(x0,0):min(x1,im.shape[1])]
    if sub.size == 0: return None
    if py0 or px0 or py1 or px1:
        if zero_pad:
            sub = cv2.copyMakeBorder(sub, py0, py1, px0, px1, cv2.BORDER_CONSTANT, value=0)
        else:
            if py0: sub = np.vstack([fill(sub[:24,:], (py0, sub.shape[1])), sub])
            if py1: sub = np.vstack([sub, fill(sub[-24:,:], (py1, sub.shape[1]))])
            if px0: sub = np.hstack([fill(sub[:,:24], (sub.shape[0], px0)), sub])
            if px1: sub = np.hstack([sub, fill(sub[:,-24:], (sub.shape[0], px1))])
    return cv2.resize(sub, (S, S), interpolation=interp)

# ---------- DICOM -> 8-bit, matched to CBIS dynamic range --------------
def read_dicom_8bit(path):
    ds = pydicom.dcmread(path)
    a = ds.pixel_array.astype(np.float32)
    if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
        a = a.max() - a                                  # invert so breast is bright
    body = a[a > a.min() + 1e-6]                         # ignore the air background
    if body.size < 1000: body = a.ravel()
    lo, hi = np.percentile(body, [1.0, 99.5])
    if hi <= lo: lo, hi = float(a.min()), float(a.max() + 1e-6)
    return np.clip((a - lo) / (hi - lo) * 255.0, 0, 255).astype(np.uint8), ds

rows, skip, diag = [], {}, {"cov_t":[], "cov_w":[], "side_t":[], "pad":[], "shape":[]}
t0 = time.time()
for i, r in man.iterrows():
    if (i+1) % 25 == 0: print("   %3d/%d  (%.0fs)" % (i+1, len(man), time.time()-t0), flush=True)
    polys = pts_map.get(r["img_id"], [])
    if not polys: skip["no polygon"] = skip.get("no polygon",0)+1; continue
    try:
        full, ds = read_dicom_8bit(r["dcm"])
    except Exception as e:
        skip["dicom read"] = skip.get("dicom read",0)+1; continue
    H, W = full.shape
    diag["shape"].append((H, W))

    gt = np.zeros((H, W), np.uint8)
    for p in polys:
        a = np.array(p, np.float64)
        a[:,0] = np.clip(a[:,0], 0, W-1); a[:,1] = np.clip(a[:,1], 0, H-1)
        cv2.fillPoly(gt, [np.round(a).astype(np.int32)], 255)   # Point_px is (x, y)
    ys, xs = np.where(gt > 127)
    if len(ys) < 50: skip["empty mask"] = skip.get("empty mask",0)+1; continue

    A = float(len(ys))
    cy, cx = 0.5*(ys.min()+ys.max()), 0.5*(xs.min()+xs.max())
    bmax = float(max(ys.max()-ys.min(), xs.max()-xs.min()) + 1)
    side_t = float(np.clip(np.sqrt(A / TARGET_COV), 1.1*bmax, 5.0*bmax))
    m = MARGIN * bmax

    # tight crop
    st = int(round(side_t))
    y0, py = place(float(ys.min()), float(ys.max()), cy, st, H, m)
    x0, px = place(float(xs.min()), float(xs.max()), cx, st, W, m)
    img_t = cut(full, y0, x0, st, cv2.INTER_AREA)
    msk_t = cut(gt,   y0, x0, st, cv2.INTER_NEAREST, zero_pad=True)
    # wide crop
    sw = int(round(side_t * WIDE_MULT))
    yw, pyw = place(float(ys.min()), float(ys.max()), cy, sw, H, m)
    xw, pxw = place(float(xs.min()), float(xs.max()), cx, sw, W, m)
    img_w = cut(full, yw, xw, sw, cv2.INTER_AREA)
    msk_w = cut(gt,   yw, xw, sw, cv2.INTER_NEAREST, zero_pad=True)
    if any(v is None for v in (img_t, msk_t, img_w, msk_w)):
        skip["cut failed"] = skip.get("cut failed",0)+1; continue
    msk_t = (msk_t > 127).astype(np.uint8)*255
    msk_w = (msk_w > 127).astype(np.uint8)*255

    key = "%s" % r["img_id"]
    pi = os.path.join(CROPT, key + "_img.png"); pm = os.path.join(CROPT, key + "_msk.png")
    wi = os.path.join(CROPW, key + "_img.png"); wm = os.path.join(CROPW, key + "_msk.png")
    for p_, im_ in [(pi,img_t),(pm,msk_t),(wi,img_w),(wm,msk_w)]: cv2.imwrite(p_, im_)

    diag["cov_t"].append((msk_t>127).mean()); diag["cov_w"].append((msk_w>127).mean())
    diag["side_t"].append(side_t); diag["pad"].append(py or px or pyw or pxw)
    rows.append(dict(img=pi, msk=pm, wimg=wi, wmsk=wm, img_id=r["img_id"],
                     patient_id=r["pid"], lesion_key="%s_%s" % (r["pid"], r["side"]),
                     side=r["side"], view=r["view"], birads=r["birads"],
                     birads_raw=r["birads_raw"], acr=r["acr"], label=int(r["label"]),
                     n_mass=int(r["n_mass"]), src_h=H, src_w=W))

d = pd.DataFrame(rows)
print("\nprocessed %d / %d  (%.1f min)" % (len(d), len(man), (time.time()-t0)/60))
if skip: print("skipped:", skip)

print("\n" + "="*76); print("VERIFICATION"); print("="*76)
print("  mask coverage, TIGHT crop   median %.1f%%   (CBIS target %.0f%%)"
      % (100*np.median(diag["cov_t"]), 100*TARGET_COV))
print("  mask coverage, WIDE crop    median %.1f%%   (expect ~%.1f%%)"
      % (100*np.median(diag["cov_w"]), 100*TARGET_COV/WIDE_MULT**2))
print("  tight window native px      median %.0f   (CBIS ~504)" % np.median(diag["side_t"]))
print("  needed tissue padding       %.1f%%" % (100*np.mean(diag["pad"])))
print("  source image size           median %s" % str(np.median(np.array(diag["shape"]), 0).astype(int)))
print("  empty masks                 %d" % int((np.array(diag["cov_t"]) < 1e-4).sum()))

print("\n  lesions (patient+side groups): %d   images: %d   malignant %.1f%%"
      % (d["lesion_key"].nunique(), len(d), 100*d["label"].mean()))
print("  both views present for: %d of %d lesions"
      % int((d.groupby("lesion_key")["view"].nunique() >= 2).sum()), )

# intensity comparison against CBIS — quantifies the domain shift
try:
    cb = pd.read_csv("/root/autodl-tmp/CBIS/unified_folds_mass.csv").sample(60, random_state=0)
    cbv = np.concatenate([cv2.imread(str(q), cv2.IMREAD_GRAYSCALE).ravel()[::37]
                          for q in cb["img"] if os.path.exists(str(q))])
    inv = np.concatenate([cv2.imread(q, cv2.IMREAD_GRAYSCALE).ravel()[::37]
                          for q in d["img"].sample(min(60, len(d)), random_state=0)])
    print("\n  intensity (crop pixels)   CBIS mean %.1f sd %.1f  |  INbreast mean %.1f sd %.1f"
          % (cbv.mean(), cbv.std(), inv.mean(), inv.std()))
    plt.figure(figsize=(6,3.4))
    plt.hist(cbv, 64, density=True, alpha=.55, label="CBIS-DDSM (film)")
    plt.hist(inv, 64, density=True, alpha=.55, label="INbreast (FFDM)")
    plt.xlabel("pixel value"); plt.ylabel("density"); plt.legend(frameon=False)
    plt.title("Domain shift: crop intensity distributions"); plt.tight_layout()
    plt.savefig(os.path.join(FIG, "domain_shift_intensity.png"), dpi=150); plt.close()
except Exception as e:
    print("  (CBIS comparison skipped: %s)" % e)

# visual check
sel = d.sample(min(4, len(d)), random_state=1).reset_index(drop=True)
fig, ax = plt.subplots(2, len(sel), figsize=(3.4*len(sel), 7))
ax = np.atleast_2d(ax)
for j in range(len(sel)):
    r = sel.iloc[j]
    for row, (ip, mp, tag) in enumerate([(r["img"], r["msk"], "tight"),
                                         (r["wimg"], r["wmsk"], "wide x1.75")]):
        im = cv2.imread(ip, cv2.IMREAD_GRAYSCALE); mk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        ax[row, j].imshow(im, cmap="gray", vmin=0, vmax=255); ax[row, j].axis("off")
        if mk is not None and (mk > 127).any():
            ax[row, j].contour(mk > 127, levels=[.5], colors="lime", linewidths=1.5)
        ax[row, j].set_title("%s\nBI-RADS %s | %s" % (tag, r["birads_raw"],
                             "MALIGNANT" if r["label"] else "benign"), fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(FIG, "inbreast_crops.png"), dpi=140); plt.close()
print("  figure: %s/inbreast_crops.png" % FIG)

d.to_csv(os.path.join(OUT, "inbreast_crops.csv"), index=False)
print("\nsaved %s/inbreast_crops.csv  (%d rows)" % (OUT, len(d)))
print("  -> next: run your segmentation model on these to get predicted masks.")

manifest rows: 107   images with points: 107
    25/107  (6s)
    50/107  (12s)
    75/107  (17s)
   100/107  (24s)

processed 107 / 107  (0.4 min)

VERIFICATION
  mask coverage, TIGHT crop   median 23.0%   (CBIS target 23%)
  mask coverage, WIDE crop    median 7.5%   (expect ~7.5%)
  tight window native px      median 508   (CBIS ~504)
  needed tissue padding       23.4%
  source image size           median [3328 2560]
  empty masks                 0

  lesions (patient+side groups): 54   images: 107   malignant 67.3%


TypeError: not enough arguments for format string

In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL I2b — CONTINUATION (run right after I2; uses objects in memory)
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

_both = int((d.groupby("lesion_key")["view"].nunique() >= 2).sum())
print("  both views present for: %d of %d lesions" % (_both, d["lesion_key"].nunique()))

# ---- domain shift: CBIS vs INbreast crop intensities ------------------
try:
    cb = pd.read_csv("/root/autodl-tmp/CBIS/unified_folds_mass.csv").sample(60, random_state=0)
    cbv = np.concatenate([cv2.imread(str(q), cv2.IMREAD_GRAYSCALE).ravel()[::37]
                          for q in cb["img"] if os.path.exists(str(q))])
    inv = np.concatenate([cv2.imread(q, cv2.IMREAD_GRAYSCALE).ravel()[::37]
                          for q in d["img"].sample(min(60, len(d)), random_state=0)])
    print("\n  crop intensity   CBIS mean %.1f sd %.1f   |   INbreast mean %.1f sd %.1f"
          % (cbv.mean(), cbv.std(), inv.mean(), inv.std()))
    plt.figure(figsize=(6, 3.4))
    plt.hist(cbv, 64, density=True, alpha=.55, label="CBIS-DDSM (digitised film)")
    plt.hist(inv, 64, density=True, alpha=.55, label="INbreast (FFDM)")
    plt.xlabel("pixel value"); plt.ylabel("density"); plt.legend(frameon=False)
    plt.title("Domain shift: crop intensity distributions"); plt.tight_layout()
    plt.savefig(os.path.join(FIG, "domain_shift_intensity.png"), dpi=150); plt.close()
    print("  figure: %s/domain_shift_intensity.png" % FIG)
except Exception as e:
    print("  (CBIS comparison skipped: %s)" % e)

# ---- visual check ------------------------------------------------------
sel = d.sample(min(4, len(d)), random_state=1).reset_index(drop=True)
fig, ax = plt.subplots(2, len(sel), figsize=(3.4*len(sel), 7))
ax = np.atleast_2d(ax)
for j in range(len(sel)):
    r = sel.iloc[j]
    for row, (ip, mp, tag) in enumerate([(r["img"], r["msk"], "tight"),
                                         (r["wimg"], r["wmsk"], "wide x1.75")]):
        im = cv2.imread(ip, cv2.IMREAD_GRAYSCALE); mk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        ax[row, j].imshow(im, cmap="gray", vmin=0, vmax=255); ax[row, j].axis("off")
        if mk is not None and (mk > 127).any():
            ax[row, j].contour(mk > 127, levels=[.5], colors="lime", linewidths=1.5)
        ax[row, j].set_title("%s\nBI-RADS %s | %s" % (tag, r["birads_raw"],
                             "MALIGNANT" if r["label"] else "benign"), fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(FIG, "inbreast_crops.png"), dpi=140); plt.close()
print("  figure: %s/inbreast_crops.png" % FIG)

d.to_csv(os.path.join(OUT, "inbreast_crops.csv"), index=False)
print("\nsaved %s/inbreast_crops.csv  (%d rows)" % (OUT, len(d)))

  both views present for: 52 of 54 lesions

  crop intensity   CBIS mean 131.3 sd 41.2   |   INbreast mean 146.3 sd 52.9
  figure: /root/autodl-tmp/INB/figures/domain_shift_intensity.png
  figure: /root/autodl-tmp/INB/figures/inbreast_crops.png

saved /root/autodl-tmp/INB/inbreast_crops.csv  (107 rows)


In [9]:
# ══════════════════════════════════════════════════════════════════════
# CELL I3 — EXTERNAL SEGMENTATION ON INbreast
#   Loads your 5 DS-Attn-UNet+ASPP fold checkpoints, runs them on the
#   107 INbreast crops, reports external Dice against CBIS's 0.900.
#   Also writes predicted masks at 512 for the classifier stage.
#   GPU, ~1 min.
# ══════════════════════════════════════════════════════════════════════
import os, glob, time
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
cv2.setNumThreads(0)

CBIS = "/root/autodl-tmp/CBIS"
OUT  = "/root/autodl-tmp/INB"
PRED = os.path.join(OUT, "predmasks"); os.makedirs(PRED, exist_ok=True)
FIG  = os.path.join(OUT, "figures");   os.makedirs(FIG, exist_ok=True)
DEV  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG, THR = 256, 0.5

# ---------- architecture: identical to Phase 2 -------------------------
def cb(i, o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class ASPP(nn.Module):
    def __init__(s,i,o):
        super().__init__()
        s.b0=nn.Sequential(nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b1=nn.Sequential(nn.Conv2d(i,o,3,padding=6,dilation=6),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b2=nn.Sequential(nn.Conv2d(i,o,3,padding=12,dilation=12),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b3=nn.Sequential(nn.Conv2d(i,o,3,padding=18,dilation=18),nn.BatchNorm2d(o),nn.ReLU(True))
        s.gp=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.proj=nn.Sequential(nn.Conv2d(o*5,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(s,x):
        g=F.interpolate(s.gp(x),size=x.shape[2:],mode="bilinear",align_corners=False)
        return s.proj(torch.cat([s.b0(x),s.b1(x),s.b2(x),s.b3(x),g],1))
class DSAttnUNet(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=ASPP(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out=nn.Conv2d(b,2,1)
        s.ds2=nn.Conv2d(b*2,2,1); s.ds3=nn.Conv2d(b*4,2,1); s.ds4=nn.Conv2d(b*8,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        main=s.out(d1)
        if s.training: return main, s.ds2(d2), s.ds3(d3), s.ds4(d4)
        return main

CKPTS = sorted(glob.glob(os.path.join(CBIS, "seg_dsaspp_mass_fold*.pth")))
print("checkpoints found: %d" % len(CKPTS))
for c in CKPTS: print("   ", os.path.basename(c))
assert CKPTS, "no seg_dsaspp_mass_fold*.pth in %s" % CBIS

# ---------- data -------------------------------------------------------
d = pd.read_csv(os.path.join(OUT, "inbreast_crops.csv"))
print("\nINbreast crops: %d images | %d lesions | malignant %.1f%%"
      % (len(d), d["lesion_key"].nunique(), 100*d["label"].mean()))
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

class INBDS(Dataset):
    def __init__(s, df): s.df = df.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s, i):
        r = s.df.iloc[i]
        im = cv2.imread(r["img"], cv2.IMREAD_GRAYSCALE)
        mk = cv2.imread(r["msk"], cv2.IMREAD_GRAYSCALE)
        if im is None: im = np.zeros((IMG,IMG), np.uint8)
        if mk is None: mk = np.zeros_like(im)
        if im.shape != (IMG,IMG): im = cv2.resize(im, (IMG,IMG))
        if mk.shape != (IMG,IMG): mk = cv2.resize(mk, (IMG,IMG), interpolation=cv2.INTER_NEAREST)
        im = _clahe.apply(im)
        return (torch.from_numpy(im.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy((mk>127).astype(np.float32)), i)

ld = DataLoader(INBDS(d), batch_size=16, shuffle=False, num_workers=0)

# ---------- run every fold model ---------------------------------------
probs = np.zeros((len(CKPTS), len(d), IMG, IMG), np.float32)
gts   = np.zeros((len(d), IMG, IMG), np.float32)
t0 = time.time()
for ci, cp in enumerate(CKPTS):
    net = DSAttnUNet().to(DEV)
    net.load_state_dict({k: v.to(DEV) for k, v in torch.load(cp, map_location="cpu").items()})
    net.eval()
    with torch.no_grad():
        for x, y, idx in ld:
            x = x.to(DEV)
            with torch.amp.autocast(device_type="cuda"):
                o = net(x)
            p = torch.softmax(o.float(), 1)[:, 1].cpu().numpy()
            for j, gi in enumerate(idx.numpy()):
                probs[ci, gi] = p[j]
                if ci == 0: gts[gi] = y[j].numpy()
    del net; torch.cuda.empty_cache()
    print("  fold %d done (%.0fs)" % (ci, time.time()-t0), flush=True)

def dice(pred, gt):
    tp = (pred*gt).sum(); fp = (pred*(1-gt)).sum(); fn = ((1-pred)*gt).sum()
    return (2*tp+1) / (2*tp+fp+fn+1)

per_fold = []
for ci in range(len(CKPTS)):
    per_fold.append(np.mean([dice((probs[ci,i] > THR).astype(np.float32), gts[i])
                             for i in range(len(d))]))
ens = probs.mean(0)
d["dice"] = [dice((ens[i] > THR).astype(np.float32), gts[i]) for i in range(len(d))]

print("\n" + "="*76); print("EXTERNAL SEGMENTATION — INbreast (never seen in training)"); print("="*76)
for ci, v in enumerate(per_fold): print("  fold %d model alone      Dice %.4f" % (ci, v))
print("  mean of the 5 models    Dice %.4f ± %.4f" % (np.mean(per_fold), np.std(per_fold)))
print("  5-model ENSEMBLE        Dice %.4f   <-- report this" % d["dice"].mean())
print("  median                  %.4f   IQR %.4f-%.4f"
      % (d["dice"].median(), d["dice"].quantile(.25), d["dice"].quantile(.75)))
print("\n  CBIS-DDSM internal      Dice 0.900")
print("  external drop           %+.4f" % (d["dice"].mean() - 0.900))
print("\n  complete failures (Dice < 0.10): %d of %d (%.1f%%)"
      % (int((d.dice < .10).sum()), len(d), 100*(d.dice < .10).mean()))
print("  good (Dice >= 0.70)            : %d of %d (%.1f%%)"
      % (int((d.dice >= .70).sum()), len(d), 100*(d.dice >= .70).mean()))

for col, nm in [("label","class"), ("acr","ACR density"), ("view","view")]:
    if col not in d.columns: continue
    t = d.groupby(col)["dice"].agg(n="size", mean="mean", median="median").round(4).reset_index()
    if col == "label": t[col] = np.where(t[col]==1, "malignant", "benign")
    print("\n  --- Dice by %s ---" % nm); print(t.to_string(index=False))

# ---------- save predicted masks at 512 for the classifier -------------
for i, r in d.iterrows():
    m = (ens[i] > THR).astype(np.uint8) * 255
    cv2.imwrite(os.path.join(PRED, "%s_pred.png" % r["key"]),
                cv2.resize(m, (512,512), interpolation=cv2.INTER_NEAREST))
d["predmask"] = d["key"].apply(lambda q: os.path.join(PRED, "%s_pred.png" % q))
d.to_csv(os.path.join(OUT, "inbreast_crops.csv"), index=False)
print("\n  predicted masks -> %s" % PRED)

# ---------- figure ------------------------------------------------------
o = d.sort_values("dice").reset_index(drop=True)
picks = [("worst",0), ("25th",len(o)//4), ("median",len(o)//2), ("75th",3*len(o)//4), ("best",len(o)-1)]
fig, ax = plt.subplots(1, len(picks), figsize=(3.2*len(picks), 3.6))
for j,(tag,i) in enumerate(picks):
    r = o.iloc[i]
    im = cv2.imread(r["img"], cv2.IMREAD_GRAYSCALE)
    gt = cv2.imread(r["msk"], cv2.IMREAD_GRAYSCALE)
    pm = cv2.imread(r["predmask"], cv2.IMREAD_GRAYSCALE)
    ax[j].imshow(im, cmap="gray"); ax[j].axis("off")
    if gt is not None and (gt>127).any(): ax[j].contour(gt>127, levels=[.5], colors="#2ecc71", linewidths=1.6)
    if pm is not None and (pm>127).any(): ax[j].contour(pm>127, levels=[.5], colors="#e74c3c",
                                                        linewidths=1.6, linestyles="--")
    ax[j].set_title("%s\nDice %.3f" % (tag, r["dice"]), fontsize=9.5)
plt.suptitle("External segmentation on INbreast — green: radiologist, red: predicted", fontsize=11)
plt.tight_layout(rect=[0,0,1,0.93])
plt.savefig(os.path.join(FIG, "inbreast_segmentation.png"), dpi=150); plt.close()
print("  figure: %s/inbreast_segmentation.png" % FIG)

checkpoints found: 5
    seg_dsaspp_mass_fold0.pth
    seg_dsaspp_mass_fold1.pth
    seg_dsaspp_mass_fold2.pth
    seg_dsaspp_mass_fold3.pth
    seg_dsaspp_mass_fold4.pth

INbreast crops: 116 images | 54 lesions | malignant 64.7%
  fold 0 done (1s)
  fold 1 done (2s)
  fold 2 done (2s)
  fold 3 done (3s)
  fold 4 done (4s)

EXTERNAL SEGMENTATION — INbreast (never seen in training)
  fold 0 model alone      Dice 0.8606
  fold 1 model alone      Dice 0.8721
  fold 2 model alone      Dice 0.8484
  fold 3 model alone      Dice 0.8831
  fold 4 model alone      Dice 0.8577
  mean of the 5 models    Dice 0.8644 ± 0.0120
  5-model ENSEMBLE        Dice 0.8802   <-- report this
  median                  0.8914   IQR 0.8510-0.9255

  CBIS-DDSM internal      Dice 0.900
  external drop           -0.0198

  complete failures (Dice < 0.10): 0 of 116 (0.0%)
  good (Dice >= 0.70)            : 115 of 116 (99.1%)

  --- Dice by class ---
    label  n   mean  median
   benign 41 0.8867  0.9024
malignant 7

In [6]:
# ══════════════════════════════════════════════════════════════════════
# CELL I3b — WHERE DOES EXTERNAL SEGMENTATION FAIL?   ~10 s
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd, cv2
pd.set_option("display.width", 200)

OUT = "/root/autodl-tmp/INB"
d = pd.read_csv(os.path.join(OUT, "inbreast_crops.csv"))
print("n=%d   mean Dice %.4f   median %.4f" % (len(d), d.dice.mean(), d.dice.median()))

print("\n--- distribution ---")
for lo, hi in [(0,.10),(.10,.30),(.30,.50),(.50,.70),(.70,.85),(.85,1.01)]:
    m = (d.dice >= lo) & (d.dice < hi)
    print("   Dice %.2f-%.2f : %3d images (%.1f%%)" % (lo, hi, m.sum(), 100*m.mean()))

bad = d[d.dice < 0.50]
print("\n--- the %d failures (Dice < 0.50) ---" % len(bad))
if len(bad):
    print(bad[["img_id","view","side","birads_raw","acr","label","dice","n_mass"]]
          .sort_values("dice").to_string(index=False))
    print("\n  mean Dice EXCLUDING these %d: %.4f" % (len(bad), d[d.dice >= .50].dice.mean()))

for col, nm in [("label","class"),("acr","ACR density"),("view","view"),("birads_raw","BI-RADS")]:
    if col not in d.columns: continue
    t = d.groupby(col)["dice"].agg(n="size", mean="mean", median="median").round(3).reset_index()
    if col == "label": t[col] = np.where(t[col]==1, "malignant", "benign")
    print("\n--- by %s ---" % nm); print(t.to_string(index=False))

# does lesion size explain the failures?
areas = []
for _, r in d.iterrows():
    m = cv2.imread(r["msk"], cv2.IMREAD_GRAYSCALE)
    areas.append(float((m > 127).mean()) if m is not None else np.nan)
d["gt_cov"] = areas
q = pd.qcut(d.gt_cov.rank(method="first"), 4, labels=["smallest","Q2","Q3","largest"])
print("\n--- by lesion size (mask coverage of crop) ---")
print(d.groupby(q)["dice"].agg(n="size", mean="mean", median="median").round(3).to_string())
print("\n  correlation size vs Dice: %.3f" % d[["gt_cov","dice"]].corr().iloc[0,1])

n=107   mean Dice 0.8421   median 0.8873

--- distribution ---
   Dice 0.00-0.10 :   3 images (2.8%)
   Dice 0.10-0.30 :   1 images (0.9%)
   Dice 0.30-0.50 :   1 images (0.9%)
   Dice 0.50-0.70 :   1 images (0.9%)
   Dice 0.70-0.85 :  26 images (24.3%)
   Dice 0.85-1.01 :  75 images (70.1%)

--- the 5 failures (Dice < 0.50) ---
  img_id view side birads_raw  acr  label     dice  n_mass
20586908   CC    R          2    2      0 0.000083       2
20586960  MLO    R          2    2      0 0.000096       2
51049107   CC    L          2    3      0 0.000104       3
22580341   CC    R          2    3      0 0.201068       2
22580367   CC    L          2    3      0 0.429479       2

  mean Dice EXCLUDING these 5: 0.8772

--- by class ---
    label  n  mean  median
   benign 35 0.776   0.891
malignant 72 0.874   0.887

--- by ACR density ---
 acr  n  mean  median
   1 42 0.872   0.888
   2 36 0.828   0.880
   3 21 0.785   0.876
   4  8 0.898   0.912

--- by view ---
view  n  mean  median
  CC

/tmp/ipykernel_1928/287793598.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(d.groupby(q)["dice"].agg(n="size", mean="mean", median="median").round(3).to_string())


In [7]:
# ══════════════════════════════════════════════════════════════════════
# CELL I2-FIX — ONE CROP PER MASS CONTOUR (was: merged per image)
#   Fixes the 5 multi-mass failures. Same geometry, same normalisation.
#   CPU only, ~30 s.  Overwrites crops_tight / crops_wide.
# ══════════════════════════════════════════════════════════════════════
import os, time, shutil
import numpy as np, pandas as pd, cv2, pydicom
cv2.setNumThreads(0)

OUT   = "/root/autodl-tmp/INB"
CROPT = os.path.join(OUT, "crops_tight"); CROPW = os.path.join(OUT, "crops_wide")
for p in (CROPT, CROPW):
    shutil.rmtree(p, ignore_errors=True); os.makedirs(p, exist_ok=True)
S, TARGET_COV, WIDE_MULT, MARGIN = 512, 0.23, 1.75, 0.10
rng = np.random.default_rng(0)

man = pd.read_csv(os.path.join(OUT, "inbreast_manifest.csv")); man["img_id"] = man["img_id"].astype(str)
pts_map = np.load(os.path.join(OUT, "inbreast_mass_points.npy"), allow_pickle=True).item()

def place(lo_o, hi_o, c, s, lim, m):
    want = c - s/2.0; lo, hi = hi_o + m - s, lo_o - m
    if lo > hi: return int(round(want)), True
    lo2, hi2 = max(lo, 0.0), min(hi, lim - s)
    if lo2 > hi2: return int(round(min(max(want, lo), hi))), True
    return int(round(min(max(want, lo2), hi2))), False
def fill(st, shape):
    mu, sd = float(np.median(st)), float(st.std())
    return np.clip(rng.normal(mu, max(0.5*sd, 1.0), shape), 0, 255).astype(np.uint8)
def cut(im, y0, x0, s, interp, zero_pad=False):
    y1, x1 = y0+s, x0+s
    py0, px0 = max(0,-y0), max(0,-x0)
    py1, px1 = max(0, y1-im.shape[0]), max(0, x1-im.shape[1])
    sub = im[max(y0,0):min(y1,im.shape[0]), max(x0,0):min(x1,im.shape[1])]
    if sub.size == 0: return None
    if py0 or px0 or py1 or px1:
        if zero_pad:
            sub = cv2.copyMakeBorder(sub, py0, py1, px0, px1, cv2.BORDER_CONSTANT, value=0)
        else:
            if py0: sub = np.vstack([fill(sub[:24,:], (py0, sub.shape[1])), sub])
            if py1: sub = np.vstack([sub, fill(sub[-24:,:], (py1, sub.shape[1]))])
            if px0: sub = np.hstack([fill(sub[:,:24], (sub.shape[0], px0)), sub])
            if px1: sub = np.hstack([sub, fill(sub[:,-24:], (sub.shape[0], px1))])
    return cv2.resize(sub, (S,S), interpolation=interp)
def read_dicom_8bit(path):
    ds = pydicom.dcmread(path); a = ds.pixel_array.astype(np.float32)
    if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1": a = a.max() - a
    body = a[a > a.min() + 1e-6]
    if body.size < 1000: body = a.ravel()
    lo, hi = np.percentile(body, [1.0, 99.5])
    if hi <= lo: lo, hi = float(a.min()), float(a.max()+1e-6)
    return np.clip((a-lo)/(hi-lo)*255.0, 0, 255).astype(np.uint8)

rows, skip, cov = [], {}, []
t0 = time.time()
for i, r in man.iterrows():
    if (i+1) % 25 == 0: print("   %3d/%d (%.0fs)" % (i+1, len(man), time.time()-t0), flush=True)
    polys = pts_map.get(r["img_id"], [])
    if not polys: continue
    try: full = read_dicom_8bit(r["dcm"])
    except Exception: skip["dicom"] = skip.get("dicom",0)+1; continue
    H, W = full.shape
    for k, p in enumerate(polys):                       # <-- ONE CROP PER CONTOUR
        gt = np.zeros((H,W), np.uint8)
        a = np.array(p, np.float64)
        a[:,0] = np.clip(a[:,0], 0, W-1); a[:,1] = np.clip(a[:,1], 0, H-1)
        cv2.fillPoly(gt, [np.round(a).astype(np.int32)], 255)
        ys, xs = np.where(gt > 127)
        if len(ys) < 50: skip["tiny contour"] = skip.get("tiny contour",0)+1; continue
        A = float(len(ys))
        cy, cx = 0.5*(ys.min()+ys.max()), 0.5*(xs.min()+xs.max())
        bmax = float(max(ys.max()-ys.min(), xs.max()-xs.min()) + 1)
        side_t = float(np.clip(np.sqrt(A/TARGET_COV), 1.1*bmax, 5.0*bmax))
        m = MARGIN * bmax
        st = int(round(side_t))
        y0, _ = place(float(ys.min()), float(ys.max()), cy, st, H, m)
        x0, _ = place(float(xs.min()), float(xs.max()), cx, st, W, m)
        it = cut(full, y0, x0, st, cv2.INTER_AREA); mt = cut(gt, y0, x0, st, cv2.INTER_NEAREST, True)
        sw = int(round(side_t*WIDE_MULT))
        yw, _ = place(float(ys.min()), float(ys.max()), cy, sw, H, m)
        xw, _ = place(float(xs.min()), float(xs.max()), cx, sw, W, m)
        iw = cut(full, yw, xw, sw, cv2.INTER_AREA); mw = cut(gt, yw, xw, sw, cv2.INTER_NEAREST, True)
        if any(v is None for v in (it, mt, iw, mw)): skip["cut"] = skip.get("cut",0)+1; continue
        mt = (mt>127).astype(np.uint8)*255; mw = (mw>127).astype(np.uint8)*255
        key = "%s_%d" % (r["img_id"], k)
        pi, pm = os.path.join(CROPT, key+"_img.png"), os.path.join(CROPT, key+"_msk.png")
        wi, wm = os.path.join(CROPW, key+"_img.png"), os.path.join(CROPW, key+"_msk.png")
        for q, im_ in [(pi,it),(pm,mt),(wi,iw),(wm,mw)]: cv2.imwrite(q, im_)
        cov.append((mt>127).mean())
        rows.append(dict(img=pi, msk=pm, wimg=wi, wmsk=wm, img_id=r["img_id"], inst=k, key=key,
                         patient_id=r["pid"], lesion_key="%s_%s" % (r["pid"], r["side"]),
                         side=r["side"], view=r["view"], birads=r["birads"],
                         birads_raw=r["birads_raw"], acr=r["acr"], label=int(r["label"])))

d = pd.DataFrame(rows)
print("\ninstances: %d  (was 107 merged images; expect ~116)" % len(d))
if skip: print("skipped:", skip)
print("  images %d | lesions %d | patients %d | malignant %.1f%%"
      % (d.img_id.nunique(), d.lesion_key.nunique(), d.patient_id.nunique(), 100*d.label.mean()))
cov = np.array(cov)
print("\n  mask coverage of tight crop: median %.1f%%  min %.1f%%  (target 23%%)"
      % (100*np.median(cov), 100*cov.min()))
print("  crops below 10%% coverage : %d   <-- must be 0 now" % int((cov < .10).sum()))
d.to_csv(os.path.join(OUT, "inbreast_crops.csv"), index=False)
print("\nsaved %s/inbreast_crops.csv  -> now RE-RUN CELL I3" % OUT)

    25/107 (6s)
    50/107 (11s)
    75/107 (17s)
   100/107 (24s)

instances: 116  (was 107 merged images; expect ~116)
  images 107 | lesions 54 | patients 50 | malignant 64.7%

  mask coverage of tight crop: median 23.0%  min 22.8%  (target 23%)
  crops below 10% coverage : 0   <-- must be 0 now

saved /root/autodl-tmp/INB/inbreast_crops.csv  -> now RE-RUN CELL I3


In [10]:
# ══════════════════════════════════════════════════════════════════════
# CELL I4a — DO ANY CLASSIFIER CHECKPOINTS EXIST?   ~10 s
# ══════════════════════════════════════════════════════════════════════
import os, glob, torch
D = "/root/autodl-tmp"
files = sorted(glob.glob(os.path.join(D, "**", "*.pt"), recursive=True) +
               glob.glob(os.path.join(D, "**", "*.pth"), recursive=True))
print("checkpoint files found: %d\n" % len(files))
for f in files:
    try:
        sd = torch.load(f, map_location="cpu")
        if not isinstance(sd, dict): print("  %-58s (not a state_dict)" % os.path.basename(f)); continue
        ks = list(sd.keys())
        dn  = any(k.startswith(("bt.","bw.","b.","features.","denseblock")) or "denseblock" in k for k in ks)
        seg = any(k.startswith(("e1.","bn.b0","u4.","a4.")) for k in ks)
        kind = "SEGMENTATION (DS-Attn-UNet)" if seg else ("CLASSIFIER (DenseNet-like)" if dn else "unknown")
        two  = " [TWO-STREAM: bt.+bw.]" if (any(k.startswith("bt.") for k in ks) and
                                            any(k.startswith("bw.") for k in ks)) else ""
        print("  %-58s %6.1f MB  %d tensors  -> %s%s"
              % (os.path.relpath(f, D), os.path.getsize(f)/1e6, len(ks), kind, two))
    except Exception as e:
        print("  %-58s unreadable (%s)" % (os.path.relpath(f, D), str(e)[:40]))
print("\nWanted: a CLASSIFIER entry, ideally TWO-STREAM. If only SEGMENTATION appears,")
print("the classifier must be retrained with checkpoint saving before external testing.")

checkpoint files found: 202

  CBIS/ablate_plain_cbis_calc.pth                              31.5 MB  220 tensors  -> SEGMENTATION (DS-Attn-UNet)
  CBIS/ablate_plain_cbis_mass.pth                              31.5 MB  220 tensors  -> SEGMENTATION (DS-Attn-UNet)
  CBIS/ablate_plain_inbreast.pth                               31.5 MB  220 tensors  -> SEGMENTATION (DS-Attn-UNet)
  CBIS/al_unet_round_1.pth                                     97.9 MB  278 tensors  -> unknown
  CBIS/al_unet_round_10.pth                                    97.9 MB  278 tensors  -> unknown
  CBIS/al_unet_round_11.pth                                    97.9 MB  278 tensors  -> unknown
  CBIS/al_unet_round_12.pth                                    97.9 MB  278 tensors  -> unknown
  CBIS/al_unet_round_13.pth                                    97.9 MB  278 tensors  -> unknown
  CBIS/al_unet_round_2.pth                                     97.9 MB  278 tensors  -> unknown
  CBIS/al_unet_round_3.pth                     

In [11]:
# ══════════════════════════════════════════════════════════════════════
# CELL I4 — TWO-STREAM TRAINED ON ALL CBIS, CHECKPOINTS SAVED
#   For external validation on INbreast. 12% patient-grouped val split
#   for early stopping only. 2 seeds. ~1 h.
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
cv2.setNumThreads(0)

D, LES = "/root/autodl-tmp/CBIS", "mass"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True; torch.backends.cuda.matmul.allow_tf32 = True

QUICK_TEST = False          # True -> 1 seed, 3 epochs (~8 min) just to check it runs
SEEDS = [11, 22]
ST, SW, BATCH = 512, 384, 8
EPOCHS, FREEZE = 20, 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W, MULT, PATIENCE, ATT = 1e-4, 2.0, 0.3, 3, 7, 2.0
if QUICK_TEST: SEEDS, EPOCHS, FREEZE, MULT = [11], 3, 1, 1

WIDE = os.path.join(D, "crops_wide_%s" % LES); PM = os.path.join(D, "predmasks_%s" % LES)
d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int)
d["stem"]  = d["img"].apply(lambda p: os.path.basename(str(p)).replace("_img.png",""))
d["tmask"] = d["stem"].apply(lambda s: os.path.join(PM,   s+"_pred.png"))
d["wimg"]  = d["stem"].apply(lambda s: os.path.join(WIDE, s+"_img.png"))
d["wmask"] = d["stem"].apply(lambda s: os.path.join(WIDE, s+"_pred.png"))
assert (~d["wimg"].apply(os.path.exists)).sum() == 0, "wide crops missing"
print("CBIS mass: %d images | %d patients | malignant %.1f%%"
      % (len(d), d.patient_id.nunique(), 100*d.label.mean()))

sgkf = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=0)
tr_i, va_i = next(sgkf.split(d, d["label"], groups=d["patient_id"]))
assert not (set(d.patient_id[tr_i]) & set(d.patient_id[va_i])), "val leak"
print("train %d images / %d patients  |  val %d images / %d patients"
      % (len(tr_i), d.patient_id[tr_i].nunique(), len(va_i), d.patient_id[va_i].nunique()))

cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
def load(p, size, mask=False):
    im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if im is None: im = np.zeros((size,size), np.uint8)
    if im.shape != (size,size):
        im = cv2.resize(im, (size,size), interpolation=cv2.INTER_NEAREST if mask else cv2.INTER_AREA)
    return (im>127).astype(np.uint8) if mask else cl.apply(im)

CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    CACHE[r["stem"]] = (load(r["img"], ST), load(r["tmask"], ST, True),
                        load(r["wimg"], SW), load(r["wmask"], SW, True))
print("cached %d in %.0fs" % (len(CACHE), time.time()-t0))

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux, meta = {}, {}
for c in ["subtlety","mass_shape","mass_margins"]:
    if c not in d.columns or d[c].notna().sum()==0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z:(z>=1)&(z<=5))
        codes, n = (v-1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary); pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([q for q in pr.unique() if q != "UNK"]); mp = {q:i for i,q in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z,-1)).astype(int).values, len(cats)
    if n > 1: aux[c], meta[c] = codes, n
AK = sorted(aux.keys()); print("helper heads:", meta)

MEAN = np.array([0.485,0.456,0.406], np.float32).reshape(3,1,1)
STD  = np.array([0.229,0.224,0.225], np.float32).reshape(3,1,1)

class DS(Dataset):
    def __init__(s, idx, aug, mult=1, tta=0):
        s.idx, s.aug, s.mult, s.tta = np.asarray(idx), aug, (mult if aug else 1), tta
    def __len__(s): return len(s.idx)*s.mult
    def __getitem__(s, i):
        j = int(s.idx[i % len(s.idx)]); r = d.iloc[j]
        ti, tm, wi, wm = [a.copy() for a in CACHE[r["stem"]]]
        if s.aug:
            fh, fv = np.random.rand()<.5, np.random.rand()<.5
            kk = np.random.randint(4); aff = np.random.rand()<.7
            ang, sc = np.random.uniform(-25,25), np.random.uniform(.9,1.12)
            itn = np.random.rand()<.5
            gg, bb = np.random.uniform(.85,1.15), np.random.uniform(-12,12)
            def T(im, mk, sz):
                if fh: im, mk = im[:,::-1], mk[:,::-1]
                if fv: im, mk = im[::-1,:], mk[::-1,:]
                if kk: im, mk = np.rot90(im,kk), np.rot90(mk,kk)
                im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
                if aff:
                    M = cv2.getRotationMatrix2D((sz/2,sz/2), ang, sc)
                    im = cv2.warpAffine(im, M, (sz,sz), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
                    mk = cv2.warpAffine(mk, M, (sz,sz), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT)
                if itn: im = np.clip(im.astype(np.float32)*gg+bb, 0, 255).astype(np.uint8)
                return im, mk
            ti, tm = T(ti, tm, ST); wi, wm = T(wi, wm, SW)
        else:
            t = s.tta
            def V(im, mk):
                if   t==1: return im[:,::-1], mk[:,::-1]
                elif t==2: return im[::-1,:], mk[::-1,:]
                elif t==3: return np.rot90(im,2), np.rot90(mk,2)
                return im, mk
            ti, tm = V(ti, tm); wi, wm = V(wi, wm)
        o = []
        for im, mk in [(ti,tm),(wi,wm)]:
            f = np.ascontiguousarray(im).astype(np.float32)/255.0
            o.append(torch.from_numpy(((np.stack([f]*3,0)-MEAN)/STD).astype(np.float32)))
            o.append(torch.from_numpy(np.ascontiguousarray(mk).astype(np.float32))[None])
        av = (np.array([aux[q][j] for q in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return o[0], o[1], o[2], o[3], torch.tensor(int(r["label"])), torch.from_numpy(av)

def backbone():
    try:  return models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
    except Exception: return models.densenet121(weights=None).features
def pool2(f, m, att):
    mm = F.interpolate(m, size=f.shape[2:], mode="bilinear", align_corners=False)
    w = 1.0 + att*mm
    return torch.cat([(f*w).sum((2,3))/(w.sum((2,3))+1e-6), f.mean((2,3))], 1)
class TwoStream(nn.Module):
    def __init__(s, am):
        super().__init__()
        s.bt, s.bw = backbone(), backbone()
        Fd = 1024*4
        s.head = nn.Sequential(nn.Linear(Fd,512), nn.BatchNorm1d(512), nn.ReLU(True),
                               nn.Dropout(0.4), nn.Linear(512,2))
        s.keys = sorted(am.keys())
        s.aux = nn.ModuleList([nn.Sequential(nn.Linear(Fd,128), nn.ReLU(True),
                                             nn.Dropout(0.3), nn.Linear(128, am[q])) for q in s.keys])
    def forward(s, xt, mt, xw, mw):
        g = torch.cat([pool2(F.relu(s.bt(xt)), mt, ATT), pool2(F.relu(s.bw(xw)), mw, ATT)], 1)
        return s.head(g), [h(g) for h in s.aux]

def focal(lg, tg, al):
    ce = F.cross_entropy(lg.float(), tg, weight=al, reduction="none")
    return ((1-torch.exp(-ce))**GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0,1,2,3] if tta else [0]):
        ld = DataLoader(DS(idx, False, 1, tta=t), batch_size=12, shuffle=False, num_workers=0)
        ps = []
        for xt, mt, xw, mw, _, _ in ld:
            with torch.amp.autocast(device_type="cuda"):
                o, _ = net(xt.to(DEV), mt.to(DEV), xw.to(DEV), mw.to(DEV))
            ps += list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot+ps
    return tot/(4 if tta else 1)

y = d["label"].values
for SEED in SEEDS:
    print("\n" + "="*66); print("SEED %d" % SEED); print("="*66)
    torch.manual_seed(SEED); np.random.seed(SEED)
    n0, n1 = float((y[tr_i]==0).sum()), float((y[tr_i]==1).sum())
    al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)
    net = TwoStream(meta).to(DEV)
    back = list(net.bt.parameters()) + list(net.bw.parameters())
    for q in back: q.requires_grad = False
    hp = [q for n_, q in net.named_parameters() if not (n_.startswith("bt.") or n_.startswith("bw."))]
    scaler = torch.amp.GradScaler()
    opt = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD); sch = None
    tl = DataLoader(DS(tr_i, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)
    best, bstate, bad, t0 = -1.0, None, 0, time.time()
    for ep in range(1, EPOCHS+1):
        if ep == FREEZE+1:
            for q in back: q.requires_grad = True
            opt = torch.optim.AdamW([{"params": back, "lr": LR_BACK},
                                     {"params": hp,   "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS-FREEZE))
        net.train()
        if ep <= FREEZE: net.bt.eval(); net.bw.eval()
        for xt, mt, xw, mw, t_, a_ in tl:
            xt, mt = xt.to(DEV, non_blocking=True), mt.to(DEV, non_blocking=True)
            xw, mw = xw.to(DEV, non_blocking=True), mw.to(DEV, non_blocking=True)
            t_, a_ = t_.to(DEV), a_.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(xt, mt, xw, mw)
                loss = focal(o, t_, al)
                if len(ax):
                    loss = loss + AUX_W*sum(F.cross_entropy(q.float(), a_[:,h], ignore_index=-1)
                                            for h, q in enumerate(ax))/len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch is not None: sch.step()
        pv = predict(net, va_i, tta=False)
        a = roc_auc_score(y[va_i], pv) if len(set(y[va_i]))>1 else 0.0
        star = ""
        if a > best:
            best, bad = a, 0; star = " *"
            bstate = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        else: bad += 1
        print("   ep %2d  val-AUC %.4f%s" % (ep, a, star), flush=True)
        if bad >= PATIENCE: print("   early stop"); break
    ck = os.path.join(D, "inb_twostream_seed%d.pth" % SEED)
    torch.save({"state_dict": bstate, "meta": meta, "aux_keys": AK,
                "val_auc": best, "ST": ST, "SW": SW, "ATT": ATT}, ck)
    print("   best val-AUC %.4f  (%.0f min)  -> %s" % (best, (time.time()-t0)/60, os.path.basename(ck)))
    del net; gc.collect(); torch.cuda.empty_cache()

print("\nDONE. Checkpoints saved. Next: Cell I5 applies them to INbreast.")

CBIS mass: 1696 images | 892 patients | malignant 46.2%
train 1483 images / 781 patients  |  val 213 images / 111 patients
cached 1696 in 17s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

SEED 11
   ep  1  val-AUC 0.6154 *
   ep  2  val-AUC 0.6979 *
   ep  3  val-AUC 0.7013 *
   ep  4  val-AUC 0.8117 *
   ep  5  val-AUC 0.8689 *
   ep  6  val-AUC 0.8689
   ep  7  val-AUC 0.8365
   ep  8  val-AUC 0.8278
   ep  9  val-AUC 0.8699 *
   ep 10  val-AUC 0.8610
   ep 11  val-AUC 0.8535
   ep 12  val-AUC 0.8557
   ep 13  val-AUC 0.8388
   ep 14  val-AUC 0.8402
   ep 15  val-AUC 0.8379
   ep 16  val-AUC 0.8476
   early stop
   best val-AUC 0.8699  (25 min)  -> inb_twostream_seed11.pth

SEED 22
   ep  1  val-AUC 0.6868 *
   ep  2  val-AUC 0.7153 *
   ep  3  val-AUC 0.7780 *
   ep  4  val-AUC 0.7998 *
   ep  5  val-AUC 0.8680 *
   ep  6  val-AUC 0.8551
   ep  7  val-AUC 0.8346
   ep  8  val-AUC 0.8565
   ep  9  val-AUC 0.8674
   ep 10  val-AUC 0.8483
   ep 11  val-AUC 0.8622
 

In [12]:
# ══════════════════════════════════════════════════════════════════════
# CELL I5 — EXTERNAL CLASSIFICATION ON INbreast
#   Projects predicted masks into the wide frame, runs the 2 saved
#   two-stream models, thresholds using CBIS val only.  GPU, ~3 min.
# ══════════════════════════════════════════════════════════════════════
import os, glob, time
import numpy as np, pandas as pd, cv2, pydicom
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, roc_curve
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
cv2.setNumThreads(0)

D, OUT = "/root/autodl-tmp/CBIS", "/root/autodl-tmp/INB"
WPRED = os.path.join(OUT, "predmasks_wide"); os.makedirs(WPRED, exist_ok=True)
FIG   = os.path.join(OUT, "figures"); os.makedirs(FIG, exist_ok=True)
DEV = torch.device("cuda")
S, TARGET_COV, WIDE_MULT, MARGIN = 512, 0.23, 1.75, 0.10
rng = np.random.default_rng(0)

CK = sorted(glob.glob(os.path.join(D, "inb_twostream_seed*.pth")))
assert CK, "no inb_twostream_seed*.pth — run Cell I4"
print("checkpoints:", [os.path.basename(c) for c in CK])
ck0 = torch.load(CK[0], map_location="cpu")
META, AK, ST, SW, ATT = ck0["meta"], ck0["aux_keys"], ck0["ST"], ck0["SW"], ck0["ATT"]
print("meta:", META, "| ST %d SW %d ATT %.1f" % (ST, SW, ATT))
for c in CK: print("   %s  CBIS val-AUC %.4f" % (os.path.basename(c), torch.load(c, map_location='cpu')['val_auc']))

# ---------- 1. project predicted masks into the WIDE frame -------------
def place(lo_o, hi_o, c, s, lim, m):
    want = c - s/2.0; lo, hi = hi_o + m - s, lo_o - m
    if lo > hi: return int(round(want))
    lo2, hi2 = max(lo, 0.0), min(hi, lim - s)
    if lo2 > hi2: return int(round(min(max(want, lo), hi)))
    return int(round(min(max(want, lo2), hi2)))
def fill(st, shape):
    mu, sd = float(np.median(st)), float(st.std())
    return np.clip(rng.normal(mu, max(0.5*sd,1.0), shape), 0, 255).astype(np.uint8)
def cut(im, y0, x0, s, interp, zero_pad=False):
    y1, x1 = y0+s, x0+s
    py0, px0 = max(0,-y0), max(0,-x0)
    py1, px1 = max(0, y1-im.shape[0]), max(0, x1-im.shape[1])
    sub = im[max(y0,0):min(y1,im.shape[0]), max(x0,0):min(x1,im.shape[1])]
    if sub.size == 0: return None
    if py0 or px0 or py1 or px1:
        if zero_pad: sub = cv2.copyMakeBorder(sub, py0, py1, px0, px1, cv2.BORDER_CONSTANT, value=0)
        else:
            if py0: sub = np.vstack([fill(sub[:24,:], (py0, sub.shape[1])), sub])
            if py1: sub = np.vstack([sub, fill(sub[-24:,:], (py1, sub.shape[1]))])
            if px0: sub = np.hstack([fill(sub[:,:24], (sub.shape[0], px0)), sub])
            if px1: sub = np.hstack([sub, fill(sub[:,-24:], (sub.shape[0], px1))])
    return cv2.resize(sub, (S,S), interpolation=interp)

d = pd.read_csv(os.path.join(OUT, "inbreast_crops.csv"))
man = pd.read_csv(os.path.join(OUT, "inbreast_manifest.csv")); man["img_id"] = man["img_id"].astype(str)
DCM = dict(zip(man["img_id"], man["dcm"]))
pts_map = np.load(os.path.join(OUT, "inbreast_mass_points.npy"), allow_pickle=True).item()
print("\nprojecting predicted masks into the wide frame for %d instances..." % len(d))

shp = {}
for iid, p in DCM.items():
    try:
        hd = pydicom.dcmread(p, stop_before_pixels=True); shp[iid] = (int(hd.Rows), int(hd.Columns))
    except Exception: pass

wp, bad = [], 0
for _, r in d.iterrows():
    iid, k = str(r["img_id"]), int(r["inst"])
    H, W = shp.get(iid, (0,0))
    poly = pts_map.get(iid, [])
    if H == 0 or k >= len(poly): wp.append(None); bad += 1; continue
    gt = np.zeros((H,W), np.uint8)
    a = np.array(poly[k], np.float64)
    a[:,0] = np.clip(a[:,0], 0, W-1); a[:,1] = np.clip(a[:,1], 0, H-1)
    cv2.fillPoly(gt, [np.round(a).astype(np.int32)], 255)
    ys, xs = np.where(gt > 127)
    if len(ys) < 50: wp.append(None); bad += 1; continue
    A = float(len(ys)); cy, cx = 0.5*(ys.min()+ys.max()), 0.5*(xs.min()+xs.max())
    bmax = float(max(ys.max()-ys.min(), xs.max()-xs.min())+1)
    side_t = float(np.clip(np.sqrt(A/TARGET_COV), 1.1*bmax, 5.0*bmax)); m = MARGIN*bmax
    st_ = int(round(side_t))
    y0 = place(float(ys.min()), float(ys.max()), cy, st_, H, m)
    x0 = place(float(xs.min()), float(xs.max()), cx, st_, W, m)
    pm = cv2.imread(r["predmask"], cv2.IMREAD_GRAYSCALE)
    if pm is None: wp.append(None); bad += 1; continue
    big = cv2.resize((pm>127).astype(np.uint8)*255, (st_, st_), interpolation=cv2.INTER_NEAREST)
    canvas = np.zeros((H,W), np.uint8)
    sy0, sx0 = max(y0,0), max(x0,0); sy1, sx1 = min(y0+st_, H), min(x0+st_, W)
    if sy1 > sy0 and sx1 > sx0:
        canvas[sy0:sy1, sx0:sx1] = big[sy0-y0:sy1-y0, sx0-x0:sx1-x0]
    sw_ = int(round(side_t*WIDE_MULT))
    yw = place(float(ys.min()), float(ys.max()), cy, sw_, H, m)
    xw = place(float(xs.min()), float(xs.max()), cx, sw_, W, m)
    mw = cut(canvas, yw, xw, sw_, cv2.INTER_NEAREST, zero_pad=True)
    if mw is None: wp.append(None); bad += 1; continue
    q = os.path.join(WPRED, "%s_pred.png" % r["key"])
    cv2.imwrite(q, (mw>127).astype(np.uint8)*255); wp.append(q)
d["wpred"] = wp
d = d[d["wpred"].notna()].reset_index(drop=True)
print("  projected %d  (failed %d)" % (len(d), bad))
covw = [ (cv2.imread(q, cv2.IMREAD_GRAYSCALE) > 127).mean() for q in d["wpred"] ]
print("  predicted-mask coverage in wide frame: median %.1f%%  (CBIS ~7.5%%)" % (100*np.median(covw)))

# ---------- 2. model -----------------------------------------------------
def backbone():
    try:  return models.densenet121(weights=None).features
    except Exception: return models.densenet121(weights=None).features
def pool2(f, m, att):
    mm = F.interpolate(m, size=f.shape[2:], mode="bilinear", align_corners=False)
    w = 1.0 + att*mm
    return torch.cat([(f*w).sum((2,3))/(w.sum((2,3))+1e-6), f.mean((2,3))], 1)
class TwoStream(nn.Module):
    def __init__(s, am):
        super().__init__()
        s.bt, s.bw = backbone(), backbone()
        Fd = 1024*4
        s.head = nn.Sequential(nn.Linear(Fd,512), nn.BatchNorm1d(512), nn.ReLU(True),
                               nn.Dropout(0.4), nn.Linear(512,2))
        s.keys = sorted(am.keys())
        s.aux = nn.ModuleList([nn.Sequential(nn.Linear(Fd,128), nn.ReLU(True),
                                             nn.Dropout(0.3), nn.Linear(128, am[q])) for q in s.keys])
    def forward(s, xt, mt, xw, mw):
        g = torch.cat([pool2(F.relu(s.bt(xt)), mt, ATT), pool2(F.relu(s.bw(xw)), mw, ATT)], 1)
        return s.head(g), [h(g) for h in s.aux]

MEAN = np.array([0.485,0.456,0.406], np.float32).reshape(3,1,1)
STD  = np.array([0.229,0.224,0.225], np.float32).reshape(3,1,1)
cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
def load(p, size, mask=False):
    im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if im is None: im = np.zeros((size,size), np.uint8)
    if im.shape != (size,size):
        im = cv2.resize(im, (size,size), interpolation=cv2.INTER_NEAREST if mask else cv2.INTER_AREA)
    return (im>127).astype(np.uint8) if mask else cl.apply(im)

class EvalDS(Dataset):
    """rows must give: tight img, tight predicted mask, wide img, wide predicted mask"""
    def __init__(s, recs, tta=0): s.r, s.tta = recs, tta
    def __len__(s): return len(s.r)
    def __getitem__(s, i):
        ti, tm, wi, wm = s.r[i]
        ti, tm = load(ti, ST), load(tm, ST, True)
        wi, wm = load(wi, SW), load(wm, SW, True)
        t = s.tta
        def V(im, mk):
            if   t==1: return im[:,::-1], mk[:,::-1]
            elif t==2: return im[::-1,:], mk[::-1,:]
            elif t==3: return np.rot90(im,2), np.rot90(mk,2)
            return im, mk
        ti, tm = V(ti, tm); wi, wm = V(wi, wm)
        o = []
        for im, mk in [(ti,tm),(wi,wm)]:
            f = np.ascontiguousarray(im).astype(np.float32)/255.0
            o.append(torch.from_numpy(((np.stack([f]*3,0)-MEAN)/STD).astype(np.float32)))
            o.append(torch.from_numpy(np.ascontiguousarray(mk).astype(np.float32))[None])
        return o[0], o[1], o[2], o[3]

@torch.no_grad()
def run_models(recs):
    acc = np.zeros(len(recs))
    for c in CK:
        sd = torch.load(c, map_location="cpu")
        net = TwoStream(sd["meta"]).to(DEV)
        net.load_state_dict({k: v.to(DEV) for k, v in sd["state_dict"].items()}); net.eval()
        tot = np.zeros(len(recs))
        for t in range(4):
            ps = []
            for xt, mt, xw, mw in DataLoader(EvalDS(recs, t), batch_size=12, shuffle=False, num_workers=0):
                with torch.amp.autocast(device_type="cuda"):
                    o, _ = net(xt.to(DEV), mt.to(DEV), xw.to(DEV), mw.to(DEV))
                ps += list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
            tot += np.array(ps)
        acc += tot/4.0
        del net; torch.cuda.empty_cache()
    return acc/len(CK)

# ---------- 3. threshold from the CBIS validation split -----------------
cb = pd.read_csv(os.path.join(D, "unified_folds_mass.csv")).reset_index(drop=True)
cb["label"] = cb["label"].astype(int)
cb["stem"] = cb["img"].apply(lambda p: os.path.basename(str(p)).replace("_img.png",""))
sgkf = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=0)
_, va_i = next(sgkf.split(cb, cb["label"], groups=cb["patient_id"]))
recs_cb = [(cb.iloc[j]["img"],
            os.path.join(D, "predmasks_mass", cb.iloc[j]["stem"] + "_pred.png"),
            os.path.join(D, "crops_wide_mass", cb.iloc[j]["stem"] + "_img.png"),
            os.path.join(D, "crops_wide_mass", cb.iloc[j]["stem"] + "_pred.png")) for j in va_i]
print("\nscoring CBIS validation (%d images) to fix the threshold..." % len(recs_cb))
p_cb = run_models(recs_cb); y_cb = cb["label"].values[va_i]
GRID = np.round(np.arange(0.02, 0.981, 0.01), 3)
accs = np.array([accuracy_score(y_cb, (p_cb > t).astype(int)) for t in GRID])
THR = float(GRID[int(np.argmax(accs))])
print("  CBIS val AUC %.4f | accuracy-optimal threshold %.2f (val acc %.1f%%)"
      % (roc_auc_score(y_cb, p_cb), THR, 100*accs.max()))

# ---------- 4. INbreast --------------------------------------------------
recs_in = list(zip(d["img"], d["predmask"], d["wimg"], d["wpred"]))
print("\nscoring INbreast (%d instances)..." % len(recs_in))
d["prob"] = run_models(recs_in)

def ci_auc(y_, p_, B=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(B):
        s = r.integers(0, len(y_), len(y_))
        if len(set(y_[s])) > 1: v.append(roc_auc_score(y_[s], p_[s]))
    return np.percentile(v, [2.5, 97.5])

yi, pi_ = d["label"].values, d["prob"].values
L = d.groupby("lesion_key").agg(y=("label","max"), p=("prob","mean")).reset_index()
a_i, a_l = roc_auc_score(yi, pi_), roc_auc_score(L.y, L.p)
lo_i, hi_i = ci_auc(yi, pi_); lo_l, hi_l = ci_auc(L.y.values, L.p.values)

pr = (L.p.values > THR).astype(int)
tn, fp, fn, tp = confusion_matrix(L.y, pr, labels=[0,1]).ravel()
print("\n" + "="*84); print("EXTERNAL CLASSIFICATION — INbreast"); print("="*84)
print("  instances %d | lesions %d | patients %d | malignant %.1f%%"
      % (len(d), len(L), d.patient_id.nunique(), 100*L.y.mean()))
print("\n  per-instance AUC  %.4f  [%.3f - %.3f]" % (a_i, lo_i, hi_i))
print("  per-lesion   AUC  %.4f  [%.3f - %.3f]   <-- report this" % (a_l, lo_l, hi_l))
print("\n  at the CBIS-derived threshold %.2f:" % THR)
print("     accuracy %.1f%%   sens %.3f   spec %.3f   FP %d   missed %d"
      % (100*accuracy_score(L.y, pr), tp/max(tp+fn,1), tn/max(tn+fp,1), fp, fn))
print("     majority-class baseline: %.1f%%" % (100*max(L.y.mean(), 1-L.y.mean())))

print("\n" + "-"*84)
print("  %-34s %-10s %-10s" % ("", "internal (CBIS)", "external (INbreast)"))
print("  %-34s %-10s %-10s" % ("segmentation Dice", "0.900", "0.880"))
print("  %-34s %-10s %-10s" % ("classification AUC", "0.9078", "%.4f" % a_l))
print("  %-34s %-10s %-10s" % ("label source", "biopsy", "BI-RADS 4-6"))
print("-"*84)

for col, nm in [("birads_raw","BI-RADS"), ("acr","ACR density"), ("view","view")]:
    if col not in d.columns: continue
    t = d.groupby(col).agg(n=("prob","size"), malig=("label","mean"),
                           mean_score=("prob","mean")).round(3).reset_index()
    print("\n--- mean predicted score by %s ---" % nm); print(t.to_string(index=False))

fpr, tpr, _ = roc_curve(L.y, L.p)
plt.figure(figsize=(5.2,5))
plt.plot(fpr, tpr, lw=2.2, color="#c0392b",
         label="INbreast external  AUC %.3f [%.2f-%.2f]" % (a_l, lo_l, hi_l))
plt.plot([0,1],[0,1], "k:", lw=.9, label="chance")
plt.axhline(0, lw=0); plt.xlabel("1 - specificity"); plt.ylabel("sensitivity")
plt.title("External validation: CBIS-trained model on INbreast\n(n=%d lesions, labels = BI-RADS 4-6)" % len(L),
          fontsize=10)
plt.legend(loc="lower right", fontsize=8.5, frameon=False); plt.grid(alpha=.25)
plt.tight_layout(); plt.savefig(os.path.join(FIG, "inbreast_roc.png"), dpi=200); plt.close()
print("\n  figure: %s/inbreast_roc.png" % FIG)
d.to_csv(os.path.join(OUT, "inbreast_predictions.csv"), index=False)
print("  saved %s/inbreast_predictions.csv" % OUT)

checkpoints: ['inb_twostream_seed11.pth', 'inb_twostream_seed22.pth']
meta: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5} | ST 512 SW 384 ATT 2.0
   inb_twostream_seed11.pth  CBIS val-AUC 0.8699
   inb_twostream_seed22.pth  CBIS val-AUC 0.8680

projecting predicted masks into the wide frame for 116 instances...
  projected 116  (failed 0)
  predicted-mask coverage in wide frame: median 7.6%  (CBIS ~7.5%)

scoring CBIS validation (213 images) to fix the threshold...
  CBIS val AUC 0.8864 | accuracy-optimal threshold 0.52 (val acc 82.2%)

scoring INbreast (116 instances)...

EXTERNAL CLASSIFICATION — INbreast
  instances 116 | lesions 54 | patients 50 | malignant 66.7%

  per-instance AUC  0.8898  [0.824 - 0.944]
  per-lesion   AUC  0.8935  [0.798 - 0.967]   <-- report this

  at the CBIS-derived threshold 0.52:
     accuracy 79.6%   sens 0.833   spec 0.722   FP 5   missed 6
     majority-class baseline: 66.7%

-----------------------------------------------------------------------

In [13]:
# ══════════════════════════════════════════════════════════════════════
# CELL I5b — THE HARD EXTERNAL SUBSET   ~20 s
#   Removes the visually obvious BI-RADS 5/6 cases. What remains is
#   BI-RADS 2/3 (benign) vs 4a/4b/4c (malignant) — the ambiguous band.
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

OUT = "/root/autodl-tmp/INB"
d = pd.read_csv(os.path.join(OUT, "inbreast_predictions.csv"))
d["br"] = d["birads_raw"].astype(str).str.strip()

def ci(y, p, B=4000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(B):
        s = r.integers(0, len(y), len(y))
        if len(set(y[s])) > 1: v.append(roc_auc_score(y[s], p[s]))
    return (np.percentile(v, 2.5), np.percentile(v, 97.5)) if v else (np.nan, np.nan)

def report(sub, tag, thr=0.52):
    if sub["label"].nunique() < 2:
        print("  %-42s only one class present" % tag); return
    L = sub.groupby("lesion_key").agg(y=("label","max"), p=("prob","mean")).reset_index()
    a = roc_auc_score(L.y, L.p); lo, hi = ci(L.y.values, L.p.values)
    pr = (L.p.values > thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(L.y, pr, labels=[0,1]).ravel()
    print("  %-42s lesions %3d (%3d malig)  AUC %.3f [%.2f-%.2f]  acc %.1f%%  sens %.2f  spec %.2f"
          % (tag, len(L), int(L.y.sum()), a, lo, hi,
             100*accuracy_score(L.y, pr), tp/max(tp+fn,1), tn/max(tn+fp,1)))

print("="*118); print("EXTERNAL PERFORMANCE BY DIFFICULTY"); print("="*118)
report(d, "ALL lesions (BI-RADS 1-3 vs 4-6)")
report(d[~d["br"].isin(["5","6"])], "HARD: 2/3 vs 4a/4b/4c  (obvious removed)")
report(d[~d["br"].isin(["6"])],     "BI-RADS 6 removed")
report(d[d["br"].isin(["3","4a"])], "BOUNDARY ONLY: 3 vs 4a")

print("\n--- score separation ---")
b = d[d["label"]==0]["prob"]; m = d[d["label"]==1]["prob"]
print("  benign  n=%3d  mean %.3f  sd %.3f" % (len(b), b.mean(), b.std()))
print("  malig   n=%3d  mean %.3f  sd %.3f" % (len(m), m.mean(), m.std()))
h = d[~d["br"].isin(["5","6"])]
bh, mh = h[h.label==0]["prob"], h[h.label==1]["prob"]
print("  hard subset: benign %.3f vs malignant %.3f  (gap %.3f)"
      % (bh.mean(), mh.mean(), mh.mean()-bh.mean()))

print("\nHow to report:")
print("  'On INbreast the model reached AUC 0.894 against BI-RADS-derived labels.")
print("   Restricting to the ambiguous band (BI-RADS 2/3 vs 4a-4c), excluding the")
print("   visually obvious BI-RADS 5/6 cases, AUC was <X> — showing that external")
print("   performance on the full cohort is partly attributable to the label")
print("   definition rather than to transfer of fine discriminative features.'")

EXTERNAL PERFORMANCE BY DIFFICULTY
  ALL lesions (BI-RADS 1-3 vs 4-6)           lesions  54 ( 36 malig)  AUC 0.894 [0.80-0.97]  acc 79.6%  sens 0.83  spec 0.72
  HARD: 2/3 vs 4a/4b/4c  (obvious removed)   lesions  29 ( 11 malig)  AUC 0.788 [0.59-0.94]  acc 72.4%  sens 0.73  spec 0.72
  BI-RADS 6 removed                          lesions  50 ( 32 malig)  AUC 0.880 [0.78-0.96]  acc 78.0%  sens 0.81  spec 0.72
  BOUNDARY ONLY: 3 vs 4a                     lesions  11 (  4 malig)  AUC 0.571 [0.12-1.00]  acc 54.5%  sens 0.25  spec 0.71

--- score separation ---
  benign  n= 41  mean 0.446  sd 0.118
  malig   n= 75  mean 0.666  sd 0.121
  hard subset: benign 0.446 vs malignant 0.592  (gap 0.146)

How to report:
  'On INbreast the model reached AUC 0.894 against BI-RADS-derived labels.
   Restricting to the ambiguous band (BI-RADS 2/3 vs 4a-4c), excluding the
   visually obvious BI-RADS 5/6 cases, AUC was <X> — showing that external
   performance on the full cohort is partly attributable to th

In [14]:
import os, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
d = pd.read_csv("/root/autodl-tmp/INB/inbreast_predictions.csv")
L = d.groupby("lesion_key").agg(y=("label","max"), p=("prob","mean")).reset_index()
g = np.round(np.arange(0.02, 0.981, 0.01), 3)
a = np.array([accuracy_score(L.y, (L.p > t).astype(int)) for t in g])
print("lesions %d | malignant %.1f%% | AUC %.4f" % (len(L), 100*L.y.mean(), roc_auc_score(L.y, L.p)))
print("accuracy with the CBIS threshold 0.52 : %.1f%%" % (100*accuracy_score(L.y, (L.p > 0.52).astype(int))))
print("best possible on INbreast (in-sample) : %.1f%%  at t=%.2f  <-- CEILING, not a result"
      % (100*a.max(), g[int(a.argmax())]))
print("gap attributable to threshold transfer: %.1f points" % (100*(a.max() - accuracy_score(L.y,(L.p>0.52).astype(int)))))

lesions 54 | malignant 66.7% | AUC 0.8935
accuracy with the CBIS threshold 0.52 : 79.6%
best possible on INbreast (in-sample) : 83.3%  at t=0.53  <-- CEILING, not a result
gap attributable to threshold transfer: 3.7 points
